# Proyecto Análisis Computacional de Datos: catálogo de plazaVea

**Curso:** Análisis Computacional de Datos (DS3021)
**Docente:** José Espinoza Melgarejo

**Integrantes:**
- Denilson Jermai Cuadros Villegas 202510232
- Enrique Eusebio Torres Chafloque 202510601
- Leo Alexander Torres Ccencho 202410078
- Valeria Bazán Meléndez 202410443
- Lucas Armando Aliaga Mena 201810550

**Fuente:** https://www.plazavea.com.pe/
**Técnica:** Selenium (renderizado) + BeautifulSoup (parseo del DOM)
**Repositorio:** [Proyecto-ACD](https://github.com/DnCuadros23/web-scraping-ACD.git)

---

## Contexto y objetivo

PlazaVea es una de las cadenas de supermercados más grandes del Perú y publica su catálogo
completo con precios vigentes. Interesa porque permite armar un dataset de precios reales
del mercado peruano: precio de venta, precio regular, descuento y marca por categoría.

**Objetivo:** construir un dataset de más de 1000 productos con sus precios extraídos
directamente del sitio, sin usar ningún endpoint de API, cumpliendo la restricción de la
actividad.

**Qué se extrae por producto:** sku, nombre, marca, categoría, precio de venta, precio
regular, indicador de precio con tarjeta Oh!, descuento calculado, URL del producto, URL
de la imagen y fecha de extracción.

**Política de nulos:** la rúbrica exige un máximo de 10% de nulos por columna en el dataset
preliminar. `precio_regular` y `descuento_pct` no son nulos reales cuando un producto no
tiene oferta activa (el precio regular es el mismo precio de venta y el descuento es 0), así
que se completan con ese valor en el extractor en vez de dejarlos como `NaN`. El precio con
tarjeta Oh! solo existe para una minoría de productos, así que se guarda como indicador
booleano (`tiene_precio_oh`) en vez de un precio numérico que sería mayormente nulo.

## Consideraciones éticas y técnicas

| Consideración | Cómo se cumple | Dónde |
|---|---|---|
| Respetar `robots.txt` | Se revisa antes de programar el scraper | Sección 2.1 |
| Ética y legalidad | Solo datos públicos de catálogo (precio, nombre, categoría); ningún dato personal | Sección 6 |
| Rendimiento | `time.sleep()` entre páginas y entre categorías | Secciones 8-9 |
| Headers | `User-Agent` realista (`UA`) en `requests` y en Selenium | Sección 1 y 3 |
| Manejo de errores | `try/except` con reintentos y espera creciente | Sección 8 |
| Almacenamiento incremental | CSV parcial por categoría antes de consolidar | Sección 9 |

## 1. Preparación del entorno

Estas son las tres librerías que vimos en clase para scraping. Aquí se usan dos:
Selenium para que el navegador ejecute el JavaScript del sitio, y BeautifulSoup para
parsear el HTML ya renderizado.

In [1]:
# Solo la primera vez. En local basta con: pip install -r requirements.txt
# !pip install selenium webdriver-manager beautifulsoup4 pandas lxml

In [ ]:
import re, time, json
from collections import Counter
from datetime import datetime
from pathlib import Path

import requests
import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

BASE = "https://www.plazavea.com.pe"
UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
      "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")  # header realista

print("librerías cargadas")

## 2. Reconocimiento del sitio

Antes de escribir el scraper hay que responder dos preguntas: qué permite el sitio y
cómo entrega los datos. Todo lo que sigue justifica la técnica elegida.

### 2.1 robots.txt

Se revisa qué rutas declara el sitio como no rastreables. Esto define el alcance ético
y técnico de la extracción.

In [3]:
# robots.txt es un archivo de texto público, no un endpoint de datos
r = requests.get(f"{BASE}/robots.txt", headers={"User-Agent": UA}, timeout=20)
print("status:", r.status_code)
print(r.text[:1500])

status: 200
User-agent: * 
Disallow: /checkout
Disallow: /sitemap/brand-1.xml
Disallow: /sitemap/brand-2.xml
Disallow: /sitemap/brand-3.xml

Sitemap: https://www.plazavea.com.pe/sitemap.xml


### 2.2 ¿El HTML estático trae los productos?

Prueba controlada: se descarga la categoría con `requests` (sin navegador) y se cuenta
cuántos precios aparecen en el HTML crudo. Si el número es cero, el catálogo se arma
con JavaScript en el navegador y `requests` + BeautifulSoup por sí solos no alcanzan.

In [4]:
url_prueba = f"{BASE}/abarrotes"

r = requests.get(url_prueba, headers={"User-Agent": UA}, timeout=30)
sopa_estatica = BeautifulSoup(r.text, "html.parser")

texto = sopa_estatica.get_text(" ", strip=True)
precios_en_html = re.findall(r"S/\s*[\d.,]+", texto)

print("status:", r.status_code, "| tamaño HTML:", len(r.text), "caracteres")
print("precios encontrados en el HTML estático:", len(precios_en_html))
print("títulos de producto encontrados:", len(sopa_estatica.select("[class*='Showcase__name']")))

status: 200 | tamaño HTML: 177544 caracteres
precios encontrados en el HTML estático: 0
títulos de producto encontrados: 0


**Conclusión del reconocimiento.** El servidor devuelve la estructura de la página
(cabecera, filtros, pie) pero la grilla de productos llega vacía: se rellena después con
JavaScript. Por eso la técnica correcta es **Selenium**, que abre un Chrome real, deja que
el sitio ejecute su JavaScript, y recién entonces se toma el HTML resultante
(`driver.page_source`) para parsearlo con **BeautifulSoup**.

No se usa ningún endpoint de API del sitio: todos los datos salen del DOM renderizado,
tal como lo ve un usuario en el navegador.

---

## 3. Navegador controlado por Selenium

`webdriver_manager` descarga el chromedriver que corresponde a la versión de Chrome
instalada, así que no hay que bajarlo a mano.

`headless=False` abre la ventana y permite ver qué está pasando — útil mientras se
depura. Para la corrida larga conviene `headless=True`.

In [ ]:
def crear_driver(headless=False, ancho=1400, alto=1000):
    opciones = Options()
    if headless:
        opciones.add_argument("--headless=new")
    opciones.add_argument(f"--window-size={ancho},{alto}")
    opciones.add_argument(f"--user-agent={UA}")  # headers: UA realista
    opciones.add_argument("--disable-blink-features=AutomationControlled")  # evita marcarse como bot
    opciones.add_argument("--lang=es-PE")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opciones)
    driver.set_page_load_timeout(60)
    return driver

driver = crear_driver(headless=False)
print("driver listo")

### 3.1 Cerrar los modales

El sitio muestra el aviso de cookies y un modal de dirección de entrega. Mientras estén
abiertos tapan el contenido y el scroll no avanza. Se intenta cerrar cada candidato y se
ignora el que no exista.

In [ ]:
def cerrar_modales(driver, espera=1.0):
    cerrados = []
    candidatos = [
        (By.ID, "onetrust-accept-btn-handler"),
        (By.CSS_SELECTOR, "button[id*='accept']"),
        (By.CSS_SELECTOR, "button[class*='accept']"),
        (By.CSS_SELECTOR, "[class*='modal'] [class*='close']"),
        (By.CSS_SELECTOR, "button[aria-label='Close']"),
    ]
    for by, sel in candidatos:
        try:
            WebDriverWait(driver, espera).until(EC.element_to_be_clickable((by, sel))).click()
            cerrados.append(sel)
            time.sleep(0.5)
        except Exception:      # manejo de errores: candidato no existe, se ignora
            continue
    return cerrados

driver.get(url_prueba)
WebDriverWait(driver, 30).until(lambda d: d.execute_script("return document.readyState") == "complete")
print("modales cerrados:", cerrar_modales(driver))
time.sleep(2)

### 3.2 Comprobación: ahora sí hay productos

Misma medición que en 2.2, pero sobre el HTML que devuelve el navegador. La diferencia
entre los dos números es la evidencia de que el sitio es dinámico.

In [7]:
sopa_render = BeautifulSoup(driver.page_source, "html.parser")
precios_render = re.findall(r"S/\s*[\d.,]+", sopa_render.get_text(" ", strip=True))

print("precios con requests (HTML estático):", len(precios_en_html))
print("precios con Selenium (HTML renderizado):", len(precios_render))

precios con requests (HTML estático): 0
precios con Selenium (HTML renderizado): 81


## 4. Identificar la tarjeta de producto

Un sitio no documenta sus clases CSS y las cambia sin avisar. En vez de asumir un
selector, se busca el contenedor que se repite una vez por producto: un elemento con
clase, que contiene un precio y cuyo texto es corto (un producto, no la página entera).

In [8]:
def candidatos_tarjeta(html, min_repeticiones=8, top=12):
    soup = BeautifulSoup(html, "html.parser")
    conteo = Counter()
    for tag in soup.find_all(["div", "li", "article", "section"]):
        clases = tag.get("class") or []
        texto = tag.get_text(" ", strip=True)
        if not clases or "S/" not in texto or len(texto) > 400:
            continue
        conteo[f"{tag.name}." + ".".join(clases)] += 1
    return [(s, n) for s, n in conteo.most_common(top) if n >= min_repeticiones]

for selector, n in candidatos_tarjeta(driver.page_source):
    print(f"{n:4d}  {selector}")

  81  div.Showcase__priceBox__row
  81  div.Showcase__priceBox__col
  58  div.HA.Showcase.Showcase--food.ga-product-item
  58  div.Showcase__content
  58  div.showcase-description
  58  div.Showcase__details
  58  div.Showcase__details__text
  58  div.Showcase__priceBox
  58  div.Showcase__salePrice
  21  div.Showcase__oldPrice.Showcase__oldPrice


El selector de arriba con el conteo más parecido al número de productos visibles es
el contenedor de la tarjeta. Se imprime una tarjeta completa para leer de dónde sale
cada campo.

In [9]:
# Reemplazar por el selector que salió en la celda anterior si es distinto
SEL_TARJETA_DETECTADO = "div.Showcase__content"

muestra = BeautifulSoup(driver.page_source, "html.parser").select(SEL_TARJETA_DETECTADO)
print("tarjetas encontradas:", len(muestra))
print(muestra[0].prettify()[:2500] if muestra else "sin coincidencias: usar otro selector")

tarjetas encontradas: 58
<div class="Showcase__content" title="Galletas Saladas FIELD Soda Paquete 6un">
 <div class="Showcase__productImage">
  <div class="tag-promoted">
   <span>
    Patrocinado
   </span>
  </div>
  <div class="Showcase__linealTags">
   <div class="Showcase__flagsImage">
   </div>
  </div>
  <div class="Showcase__octogono caso-1">
   <img alt="octogono-placeholder" class="Showcase__octogono__img" src="https://vivanda.vteximg.com.br/arquivos/Alto_sodio.png?v=123"/>
  </div>
  <a class="Showcase__link" href="https://www.plazavea.com.pe/galletas-soda-field-bolsa-6un/p">
   <figure class="Showcase__photo">
    <img alt="product image" class="showcase__image" height="184" src="https://plazavea.vteximg.com.br/arquivos/ids/31462867-184-184/20282832.jpg?v=639235240045730000" style="opacity: 1; transition: opacity 0.3s; display: block;" title="product image" width="184"/>
    <div class="Showcase__flags">
    </div>
   </figure>
  </a>
 </div>
 <div class="showcase-descript

## 5. Mapa de selectores

Cada campo tiene varios selectores en orden de preferencia: se prueba el primero y, si
no aparece, el siguiente. Así un cambio de clase en el sitio degrada el resultado en vez
de romper la corrida entera.

In [10]:
SELECTORES = {
    "tarjeta": ["div.Showcase__content", "div[class*='Showcase__content']", "div.Showcase",
                "div.shelf-item", "article[class*='product']", "li[class*='product-item']"],
    "nombre": ["a.Showcase__name", ".Showcase__name", "[class*='Showcase__name']",
               "h2 a", "[class*='productName']"],
    "marca": [".Showcase__brand", "[class*='brand']"],
    "precio_venta": [".Showcase__salePrice", "[class*='salePrice']", "[class*='sellingPrice']",
                     "[class*='bestPrice']"],
    "precio_regular": [".Showcase__oldPrice", "[class*='oldPrice']", "[class*='listPrice']", "del"],
    "precio_oh": ["[class*='ohPrice']", "[class*='priceOh']", "[class*='cardPrice']"],
    "enlace": ["a.Showcase__link", "a[href*='/p']", "a[href]"],
}

MIN_TARJETAS = 3

def seleccionar_tarjetas(soup):
    """Devuelve (tarjetas, selector): se queda con el candidato de más coincidencias."""
    mejor, mejor_sel = [], None
    for sel in SELECTORES["tarjeta"]:
        t = soup.select(sel)
        if len(t) > len(mejor):
            mejor, mejor_sel = t, sel
    return (mejor, mejor_sel) if len(mejor) >= MIN_TARJETAS else ([], None)

tarjetas, sel = seleccionar_tarjetas(sopa_render)
print("selector elegido:", sel, "| tarjetas:", len(tarjetas))

selector elegido: div.Showcase | tarjetas: 78


## 6. Extractor: de tarjeta HTML a fila del dataset

Tres detalles que importan:

- **`a_numero`**: en el sitio el precio es texto (`"S/ 1,299.90"`). En formato peruano la
  coma separa miles y el punto los decimales, así que se quita la coma antes de convertir.
- **Red de seguridad de precios**: si los selectores de precio fallan, se leen todos los `S/`
  del texto de la tarjeta y se toma el menor como precio de venta y el mayor como precio
  regular. Es preferible un dato aproximado y marcado que una columna 100% nula.
- **Red de seguridad de imágenes**: el catálogo usa *lazy loading*, así que la imagen real a
  veces no está en el atributo `src` sino en `data-src`/`data-original` o en un `srcset`. Se
  prueban varios atributos en cascada, igual que con los selectores de texto.

In [11]:
def primer_texto(nodo, candidatos):
    """Primer selector de la lista que devuelva texto no vacío."""
    for sel in candidatos:
        el = nodo.select_one(sel)
        if el:
            txt = el.get_text(" ", strip=True)
            if txt:
                return txt
    return None


def a_numero(texto):
    """'S/ 1,299.90' -> 1299.90"""
    if not texto:
        return None
    m = re.search(r"(\d[\d.,]*)", texto.replace(" ", ""))
    if not m:
        return None
    try:
        return float(m.group(1).replace(",", ""))
    except ValueError:
        return None


ATRIBUTOS_IMAGEN = ["src", "data-src", "data-original", "data-lazy-src", "data-lazy"]

def primera_imagen(tarjeta):
    """El catálogo usa lazy-loading: el src real de la foto a veces no está en
    'src' (que trae un placeholder o queda vacío) sino en un atributo data-*.
    Se prueban varios atributos en cascada antes de dar la imagen por ausente."""
    for img in tarjeta.select("img"):
        for attr in ATRIBUTOS_IMAGEN:
            val = img.get(attr)
            if val and not val.startswith("data:"):
                return val
    fuente = tarjeta.select_one("picture source[srcset]")
    if fuente and fuente.get("srcset"):
        return fuente["srcset"].split(",")[0].strip().split(" ")[0]
    return None


def extraer_tarjeta(tarjeta, categoria):
    nombre = primer_texto(tarjeta, SELECTORES["nombre"])

    enlace = None
    for s in SELECTORES["enlace"]:
        a = tarjeta.select_one(s)
        if a and a.get("href"):
            enlace = a["href"]
            break
    if enlace and enlace.startswith("/"):
        enlace = BASE + enlace

    imagen = primera_imagen(tarjeta)

    precio_venta = a_numero(primer_texto(tarjeta, SELECTORES["precio_venta"]))
    precio_regular = a_numero(primer_texto(tarjeta, SELECTORES["precio_regular"]))
    precio_oh = a_numero(primer_texto(tarjeta, SELECTORES["precio_oh"]))

    # fallback: leer los montos directamente del texto de la tarjeta
    if precio_venta is None:
        montos = [a_numero(m) for m in re.findall(r"S/\s*[\d.,]+", tarjeta.get_text(" ", strip=True))]
        montos = [m for m in montos if m]
        if montos:
            precio_venta = min(montos)
            precio_regular = precio_regular or (max(montos) if len(montos) > 1 else None)

    # Sin oferta activa el precio regular es el mismo precio de venta y el
    # descuento es 0: no son nulos reales, son la ausencia de una promoción.
    descuento = 0.0
    if precio_venta and precio_regular and precio_regular > precio_venta:
        descuento = round((1 - precio_venta / precio_regular) * 100, 2)
    else:
        precio_regular = precio_venta

    sku = None
    if enlace:
        m = re.search(r"/(\d{4,})/?p?/?$", enlace) or re.search(r"/([^/]+)/p$", enlace)
        sku = m.group(1) if m else None

    return {
        "sku": sku,
        "nombre": nombre,
        "marca": primer_texto(tarjeta, SELECTORES["marca"]),
        "categoria": categoria,
        "precio_venta": precio_venta,
        "precio_regular": precio_regular,
        "tiene_precio_oh": precio_oh is not None,
        "descuento_pct": descuento,
        "url_producto": enlace,
        "url_imagen": imagen,
        "fecha_extraccion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

### 6.1 Prueba sobre las primeras tarjetas

Se valida con pocas filas antes de correr nada grande. Si una columna sale 100% nula,
el selector está mal escrito o el campo está un nivel más adentro: se corrige acá, con
5 filas, no después con 2000.

In [12]:
prueba = pd.DataFrame([extraer_tarjeta(t, "abarrotes") for t in tarjetas[:5]])
display(prueba)
print("\nnulos por columna (%):")
print((prueba.isnull().mean() * 100).round(1))

,sku,nombre,marca,categoria,precio_venta,precio_regular,tiene_precio_oh,descuento_pct,url_producto,url_imagen,fecha_extraccion
0,None,None,None,abarrotes,None,None,False,0.0,None,None,2026-09-16 21:06:35
1,None,None,None,abarrotes,None,None,False,0.0,None,None,2026-09-16 21:06:35
2,None,None,None,abarrotes,None,None,False,0.0,None,None,2026-09-16 21:06:35
3,None,None,None,abarrotes,None,None,False,0.0,None,None,2026-09-16 21:06:35
4,None,None,None,abarrotes,None,None,False,0.0,None,None,2026-09-16 21:06:35



nulos por columna (%):
sku                 100.0
nombre              100.0
marca               100.0
categoria             0.0
precio_venta        100.0
precio_regular      100.0
tiene_precio_oh       0.0
descuento_pct         0.0
url_producto        100.0
url_imagen          100.0
fecha_extraccion      0.0
dtype: float64


## 7. Paginación y carga de imágenes

Dos problemas que se detectaron al correr la primera versión de esta sección con datos
reales:

1. **El scroll infinito tiene un techo fijo.** Insistir con `window.scrollTo` hasta 8 veces
   seguidas no pasó nunca de ~58-78 tarjetas por categoría, aunque la categoría tenga miles
   de productos (`abarrotes` declara 3275). La paginación real del catálogo es un parámetro
   de la URL (`?page=N`): navegar página por página sí trae productos nuevos cada vez.
2. **Un salto de scroll grande deja las fotos sin cargar.** Las imágenes usan *lazy loading*:
   si se salta directo al final de la página, la mayoría de `<img>` se queda con `src=""`
   porque nunca "entraron" al viewport para disparar su carga. Bajando en pasos pequeños por
   toda la página antes de leer el HTML, esto se resuelve (0% de imágenes faltantes en la
   prueba).

Por eso el motor de extracción cambia de "scroll infinito" a **paginación por URL**, con un
recorrido de scroll gradual en cada página para forzar la carga de imágenes.

In [13]:
def contar_tarjetas(driver):
    mejor = 0
    for sel in SELECTORES["tarjeta"]:
        try:
            mejor = max(mejor, len(driver.find_elements(By.CSS_SELECTOR, sel)))
        except Exception:
            continue
    return mejor if mejor >= MIN_TARJETAS else 0


def desplazar_para_cargar_imagenes(driver, paso=700, pausa=0.5):
    """Recorre toda la página en pasos pequeños (en vez de un salto directo al
    final) para que el lazy-loading de cada foto tenga tiempo de disparar su
    IntersectionObserver. Verificado: sin este paso, ~72% de las fotos quedan
    con src vacío; con scroll gradual, 0%."""
    alto = driver.execute_script("return document.body.scrollHeight")
    pos = 0
    while pos < alto:
        driver.execute_script(f"window.scrollTo(0, {pos});")
        time.sleep(pausa)
        pos += paso
        alto = driver.execute_script("return document.body.scrollHeight")


def construir_url_pagina(url_categoria, pagina, tamano_pagina=100):
    """PS es el page size del catálogo (VTEX) y page el número de página.
    Es la paginación real: a diferencia del scroll infinito (que tiene un
    techo fijo de ~60-110 tarjetas sin importar cuánto se scrollee), cada
    valor de `page` trae productos distintos."""
    separador = "&" if "?" in url_categoria else "?"
    return f"{url_categoria}{separador}PS={tamano_pagina}&page={pagina}"


# Prueba rápida: página 1 vs página 2 de abarrotes deben traer productos distintos
driver.get(construir_url_pagina(f"{BASE}/abarrotes", 1))
time.sleep(2)
desplazar_para_cargar_imagenes(driver)
print("página 1:", contar_tarjetas(driver), "tarjetas")

driver.get(construir_url_pagina(f"{BASE}/abarrotes", 2))
time.sleep(2)
desplazar_para_cargar_imagenes(driver)
print("página 2:", contar_tarjetas(driver), "tarjetas")

página 1: 108 tarjetas
página 2: 100 tarjetas


## 8. Motor de scraping con manejo de errores

El motor recorre página por página (`?page=1,2,3...`) en vez de scrollear una sola página,
porque esa es la paginación real del catálogo. Se detiene cuando:

- una página no trae ningún producto **nuevo** (se llegó al final del catálogo de la
  categoría), o
- se alcanza `max_productos`, o
- se alcanza `max_paginas` (límite de seguridad para no quedarse en un loop infinito).

Dos tipos de falla se tratan distinto:

- **Página que no carga o pierde conexión** → es temporal: se reintenta con espera creciente.
- **Cero tarjetas en la página 1** → o cambió el selector, o el sitio está bloqueando: no se
  reintenta a ciegas, se avisa y se abandona la categoría.
- **Cero tarjetas en una página > 1** → normal: es el final del catálogo, no un error.

Las filas sin nombre o sin URL de producto se descartan (banners) y se deduplica por
`url_producto` dentro de la misma categoría, porque una página puede repetir productos que
ya aparecieron destacados en otra.

In [ ]:
def scrapear_categoria(driver, url, categoria, max_productos=200, max_paginas=15,
                        intentos=3, pausa=2.0, verbose=True):
    vistos = set()
    filas = []

    for pagina in range(1, max_paginas + 1):
        url_pagina = construir_url_pagina(url, pagina)
        cargo = False

        for intento in range(1, intentos + 1):
            try:                                       # manejo de errores
                if verbose:
                    print(f"[{categoria}] página {pagina} intento {intento} -> {url_pagina}")
                driver.get(url_pagina)
                WebDriverWait(driver, 30).until(
                    lambda d: d.execute_script("return document.readyState") == "complete")
                cerrar_modales(driver)
                time.sleep(pausa)                       # rendimiento: no saturar servidor

                if contar_tarjetas(driver) == 0:
                    raise ValueError("cero tarjetas")

                cargo = True
                break

            except Exception as e:
                if pagina > 1:
                    # en páginas > 1, cero tarjetas normalmente significa que
                    # se acabó el catálogo de la categoría, no un error real
                    break
                print(f"[{categoria}] falló intento {intento}: {type(e).__name__}: {e}")
                time.sleep(pausa * intento * 2)      # reintento con espera creciente

        if not cargo:
            if pagina == 1:
                print(f"[{categoria}] se abandona la categoría: no responde ni la página 1")
            elif verbose:
                print(f"[{categoria}] fin del catálogo en la página {pagina}")
            break

        desplazar_para_cargar_imagenes(driver)

        soup = BeautifulSoup(driver.page_source, "html.parser")
        tarjetas, sel = seleccionar_tarjetas(soup)

        nuevas = 0
        for t in tarjetas:
            fila = extraer_tarjeta(t, categoria)
            if not fila["nombre"] or not fila["url_producto"]:
                continue                              # banner o espacio publicitario
            if fila["url_producto"] in vistos:
                continue                              # ya se extrajo en otra página
            vistos.add(fila["url_producto"])
            filas.append(fila)
            nuevas += 1

        if verbose:
            print(f"[{categoria}] selector={sel} | tarjetas={len(tarjetas)} | "
                  f"nuevas={nuevas} | total={len(filas)}")

        if nuevas == 0 or len(filas) >= max_productos:
            break
        time.sleep(1)                                 # rendimiento: pausa entre páginas

    return filas[:max_productos]

### 8.1 Corrida de prueba sobre una categoría

In [15]:
filas_abarrotes = scrapear_categoria(driver, f"{BASE}/abarrotes", "abarrotes", max_productos=80)

df_abarrotes = pd.DataFrame(filas_abarrotes)
print("forma:", df_abarrotes.shape)
display(df_abarrotes.head(10))

[abarrotes] página 1 intento 1 -> https://www.plazavea.com.pe/abarrotes?PS=100&page=1
[abarrotes] selector=div.Showcase | tarjetas=108 | nuevas=58 | total=58
[abarrotes] página 2 intento 1 -> https://www.plazavea.com.pe/abarrotes?PS=100&page=2
[abarrotes] selector=div.Showcase | tarjetas=100 | nuevas=48 | total=106
forma: (80, 11)


,sku,nombre,marca,categoria,precio_venta,precio_regular,tiene_precio_oh,descuento_pct,url_producto,url_imagen,fecha_extraccion
0,galletas-soda-field-bolsa-6un,Galletas Saladas FIELD Soda Paquete 6un,FIELD,abarrotes,3.50,3.90,False,10.26,https://www.plazavea.com.pe/galletas-soda-fiel...,https://vivanda.vteximg.com.br/arquivos/Alto_s...,2026-09-16 21:07:45
1,gomas-ambrosoli-cerebros-bolsa-250g,Gomas AMBROSOLI Cerebros Bolsa 250g,AMBROSOLI,abarrotes,14.50,14.50,False,0.00,https://www.plazavea.com.pe/gomas-ambrosoli-ce...,https://plazavea.vteximg.com.br/arquivos/ids/3...,2026-09-16 21:07:45
2,galleta-chokosoda-field-paquete-6un,Galletas FIELD ChokoSoda Paquete 6un,FIELD,abarrotes,6.30,7.20,False,12.50,https://www.plazavea.com.pe/galleta-chokosoda-...,https://vivanda.vteximg.com.br/arquivos/azucar...,2026-09-16 21:07:45
3,galleta-vainilla-field-paquete-6un,Galletas FIELD Vainilla Paquete 6un,FIELD,abarrotes,4.80,5.50,False,12.73,https://www.plazavea.com.pe/galleta-vainilla-f...,https://vivanda.vteximg.com.br/arquivos/Alto_a...,2026-09-16 21:07:45
4,chupetin-de-halloween-mister-pops-extreme-duo-...,Chupetín de Halloween MISTER POPS Extreme Duo ...,MISTER POPS,abarrotes,10.80,10.80,False,0.00,https://www.plazavea.com.pe/chupetin-de-hallow...,https://plazavea.vteximg.com.br/arquivos/ids/2...,2026-09-16 21:07:45
5,chupetines-mister-pops-ciclopops-halloween-bol...,Chupetines MISTER POPS Ciclopops Halloween Bol...,MISTER POPS,abarrotes,10.80,10.80,False,0.00,https://www.plazavea.com.pe/chupetines-mister-...,https://vivanda.vteximg.com.br/arquivos/Alto_a...,2026-09-16 21:07:45
6,conserva-bells-palmitos-enteros-en-salmuera-fr...,Conserva BELL'S Palmitos Enteros en Salmuera F...,BELL'S,abarrotes,7.69,7.69,False,0.00,https://www.plazavea.com.pe/conserva-bells-pal...,https://plazavea.vteximg.com.br/arquivos/ids/3...,2026-09-16 21:07:45
7,aceite-vegetal-primor-clasico-botella-900ml,Aceite Vegetal PRIMOR Clásico Botella 900ml,PRIMOR,abarrotes,8.50,9.70,True,12.37,https://www.plazavea.com.pe/aceite-vegetal-pri...,https://plazavea.vteximg.com.br/arquivos/ids/3...,2026-09-16 21:07:45
8,galletas-field-cream-crakers-bolsa-73-5g,Galletas Saladas FIELD Cream Cracker Paquete 7...,FIELD,abarrotes,2.10,2.50,False,16.00,https://www.plazavea.com.pe/galletas-field-cre...,https://plazavea.vteximg.com.br/arquivos/ids/3...,2026-09-16 21:07:45
9,galletas-field-charada-sixpack-paquete-226-8g,Galletas FIELD Charada Paquete 6un,FIELD,abarrotes,5.30,5.30,False,0.00,https://www.plazavea.com.pe/galletas-field-cha...,https://vivanda.vteximg.com.br/arquivos/azucar...,2026-09-16 21:07:45


## 9. Extracción de todas las categorías

Se recorre categoría por categoría, se guarda un parcial después de cada una (si algo
falla a mitad de camino no se pierde lo anterior) y se hace una pausa entre categorías
para no golpear el servidor.

Se pasó de 5 a 11 categorías del mismo vertical "Supermercado" (verificadas manualmente
navegando el sitio) para superar cómodamente los 1000 registros que pide la rúbrica: cada
categoría de PlazaVea tiene entre cientos y miles de productos, así que con
`max_productos=150` por categoría el total esperado es de ~1000-1600 filas.

**Nota sobre categorías nuevas:** PlazaVea reutiliza algunos slugs de URL entre el vertical
de supermercado y el de "Electro, hogar y más" (marketplace de terceros). Si una categoría
nueva cae en el vertical equivocado, sus tarjetas no calzarán con `SELECTORES["tarjeta"]`
(que es específico de las tarjetas `Showcase__*` del supermercado) y `scrapear_categoria`
la descarta sola con el error "cero tarjetas". Por eso no hace falta confiar a ciegas en el
nombre del slug: si una categoría nueva devuelve 0 filas, no se agrega al diccionario.

In [ ]:
CATEGORIAS = {
    "abarrotes":         f"{BASE}/abarrotes",
    "bebidas":           f"{BASE}/bebidas",
    "lacteos":           f"{BASE}/lacteos-y-huevos",
    "limpieza":          f"{BASE}/cuidado-del-hogar",
    "cuidado-personal":  f"{BASE}/cuidado-personal",
    "frutas-verduras":   f"{BASE}/frutas-y-verduras",
    "carnes-pescados":   f"{BASE}/carnes-aves-y-pescados",
    "desayunos":         f"{BASE}/desayunos",
    "quesos-fiambres":   f"{BASE}/quesos-y-fiambres",
    "panaderia":         f"{BASE}/panaderia-y-pasteleria",
    "congelados":        f"{BASE}/congelados",
}

Path("../data/raw").mkdir(parents=True, exist_ok=True)

todo = []
for cat, url in CATEGORIAS.items():
    filas = scrapear_categoria(driver, url, cat, max_productos=150)
    if filas:
        # almacenamiento incremental: si falla una categoría más adelante, esto ya quedó guardado
        pd.DataFrame(filas).to_csv(f"../data/raw/parcial_{cat}.csv", index=False, encoding="utf-8-sig")
        todo.extend(filas)
    time.sleep(3)   # rendimiento: pausa entre categorías

print("\ntotal de filas:", len(todo))

In [17]:
driver.quit()   # cerrar el navegador al terminar
print("navegador cerrado")

navegador cerrado


## 10. Dataset resultante y control de calidad

Tres cosas que revisar antes de entregar: tamaño (más de 1000 filas), duplicados
(un mismo producto puede aparecer en dos categorías) y nulos por columna.

In [18]:
df = pd.DataFrame(todo)

print("filas:", len(df), "| columnas:", df.shape[1])
print("cumple mínimo de 1000 filas:", len(df) >= 1000)

print("\nproductos por categoría:")
print(df.groupby("categoria").size())

print("\nduplicados por url_producto:", df["url_producto"].duplicated().sum())
df = df.drop_duplicates(subset="url_producto").reset_index(drop=True)

nulos_pct = (df.isnull().mean() * 100).round(1)
print("\nnulos por columna (%) tras quitar duplicados:")
print(nulos_pct)

excede_10pct = nulos_pct[nulos_pct > 10]
print("\ncolumnas que exceden el 10% de nulos (rúbrica exige máximo 10%):")
print(excede_10pct if not excede_10pct.empty else "ninguna")

filas: 1515 | columnas: 11
cumple mínimo de 1000 filas: True

productos por categoría:
categoria
abarrotes           150
bebidas             150
carnes-pescados     150
congelados          150
cuidado-personal    150
desayunos           150
frutas-verduras     150
lacteos             150
limpieza             15
panaderia           150
quesos-fiambres     150
dtype: int64

duplicados por url_producto: 8

nulos por columna (%) tras quitar duplicados:
sku                 0.0
nombre              0.0
marca               0.0
categoria           0.0
precio_venta        0.0
precio_regular      0.0
tiene_precio_oh     0.0
descuento_pct       0.0
url_producto        0.0
url_imagen          0.0
fecha_extraccion    0.0
dtype: float64

columnas que exceden el 10% de nulos (rúbrica exige máximo 10%):
ninguna


In [19]:
marca = datetime.now().strftime("%Y%m%d_%H%M")
ruta = f"../data/raw/plazavea_crudo_{marca}.csv"
df.to_csv(ruta, index=False, encoding="utf-8-sig")
print("guardado:", ruta)

df.describe(include="all").T


guardado: ../data/raw/plazavea_crudo_20260916_2123.csv


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
sku,1507,1507,lasagna-de-carne-antigua-taberna-queirolo-caja...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
nombre,1507,1490,Bebida de Almendras NATURE'S HEART sin Azúcar ...,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
marca,1507,276,BELL'S,118,NaN,NaN,NaN,NaN,NaN,NaN,NaN
categoria,1507,11,abarrotes,150,NaN,NaN,NaN,NaN,NaN,NaN,NaN
precio_venta,1507.0,NaN,NaN,NaN,27.668042,62.63083,0.9,6.5,11.1,19.9,999.0
precio_regular,1507.0,NaN,NaN,NaN,36.017817,96.360902,1.0,6.9,12.0,20.9,1618.0
tiene_precio_oh,1507,2,False,1391,NaN,NaN,NaN,NaN,NaN,NaN,NaN
descuento_pct,1507.0,NaN,NaN,NaN,8.019516,11.926071,0.0,0.0,0.0,13.33,72.89
url_producto,1507,1507,https://www.plazavea.com.pe/lasagna-de-carne-a...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
url_imagen,1507,1128,https://plazavea.vteximg.com.br/arquivos/origi...,127,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 11. Conclusiones

1. **El sitio es dinámico.** La prueba de la sección 2.2 lo demuestra con números: el HTML
   estático llega con la grilla vacía y el HTML renderizado por Selenium trae los precios.
   Esa medición es la que justifica la técnica, no una preferencia.

2. **Selenium renderiza, BeautifulSoup extrae.** Cada librería hace lo que hace bien:
   Selenium controla el navegador y espera a que cargue; BeautifulSoup recorre el árbol
   HTML resultante. No se usó ningún endpoint de API del sitio.

3. **Los selectores son el punto frágil.** Por eso cada campo tiene varios candidatos y el
   precio tiene un fallback por expresión regular. Un cambio de clase en el sitio degrada
   una columna en vez de tumbar toda la corrida.

4. **Limitaciones.** Los precios dependen de la zona de entrega configurada en el sitio, así
   que el dataset es una foto de una zona y un momento; por eso cada fila lleva
   `fecha_extraccion`. Los productos sin stock pueden aparecer con precio 0, lo que se
   trata en la etapa de limpieza.

**Siguiente paso:** limpieza y diccionario de datos sobre el CSV crudo generado aquí.

---

# Parte II — Limpieza y control de calidad del dataset

**Responsable de esta sección:** *(P4 — Limpieza y calidad)*

Las secciones 1–11 arman el **dataset crudo** (`data/raw/plazavea_crudo_*.csv`): 1507 filas,
11 columnas y **0 % de nulos en todas las columnas**. Ese 0 % es engañoso: no significa que
el dato esté bien, sino que ningún defecto se manifiesta como celda vacía. Esta segunda parte
detecta y corrige los defectos reales, deja registro de cada decisión en una **bitácora**, y
verifica de forma explícita los requisitos de calidad de la rúbrica.

Es la realización del "siguiente paso" que anuncia la sección 11: limpieza sobre el CSV crudo
generado por el scraping. Lee el crudo desde disco, así que se puede correr de forma
independiente sin volver a abrir el navegador.

### Requisitos de la rúbrica que se verifican aquí (*Calidad de los datos*)

| Requisito | Umbral |
|---|---|
| Volumen | **> 1000 registros** |
| Estructura | **≥ 6 atributos** |
| Nulos | **≤ 10 % por columna** |
| Duplicados | **sin duplicados evidentes** |

La comprobación de los cuatro puntos, impresa como booleanos, está en la sección 15.

## 12. Carga del crudo y diagnóstico inicial

Se toma el CSV crudo más reciente de `data/raw/` (el que acaba de generar el scraping) y se
mide el estado de partida: forma, nulos, duplicados y balance de categorías. Este diagnóstico
es la línea base contra la que se compara el resultado final.

In [1]:
import re
from pathlib import Path

import pandas as pd

# El notebook usa rutas relativas; ejecutar desde notebooks/
DIR_RAW = Path("../data/raw")
OUT = Path("../data/processed/plazavea_limpio.csv")

# Se toma el crudo consolidado más reciente (ignora los parciales por categoría)
crudos = sorted(DIR_RAW.glob("plazavea_crudo_*.csv"))
assert crudos, "No se encontró ningún plazavea_crudo_*.csv: corre primero el scraping (secciones 1-10)."
RAW = crudos[-1]

df_crudo = pd.read_csv(RAW)
print("archivo crudo:", RAW.name)
print("forma del crudo:", df_crudo.shape)
print("columnas:", list(df_crudo.columns))
df_crudo.head(3)

archivo crudo: plazavea_crudo_20260916_2123.csv
forma del crudo: (1507, 11)
columnas: ['sku', 'nombre', 'marca', 'categoria', 'precio_venta', 'precio_regular', 'tiene_precio_oh', 'descuento_pct', 'url_producto', 'url_imagen', 'fecha_extraccion']


,sku,nombre,marca,categoria,precio_venta,precio_regular,tiene_precio_oh,descuento_pct,url_producto,url_imagen,fecha_extraccion
0,galletas-soda-field-bolsa-6un,Galletas Saladas FIELD Soda Paquete 6un,FIELD,abarrotes,3.5,3.9,False,10.26,https://www.plazavea.com.pe/galletas-soda-fiel...,https://vivanda.vteximg.com.br/arquivos/Alto_s...,2026-09-16 21:08:39
1,gomas-ambrosoli-cerebros-bolsa-250g,Gomas AMBROSOLI Cerebros Bolsa 250g,AMBROSOLI,abarrotes,14.5,14.5,False,0.00,https://www.plazavea.com.pe/gomas-ambrosoli-ce...,https://plazavea.vteximg.com.br/arquivos/ids/3...,2026-09-16 21:08:39
2,galleta-chokosoda-field-paquete-6un,Galletas FIELD ChokoSoda Paquete 6un,FIELD,abarrotes,6.3,7.2,False,12.50,https://www.plazavea.com.pe/galleta-chokosoda-...,https://vivanda.vteximg.com.br/arquivos/azucar...,2026-09-16 21:08:39


### 12.1 Nulos y duplicados aparentes

`isnull` no revela nada porque el extractor rellenó los huecos (un producto sin oferta guarda
`precio_regular = precio_venta` y `descuento_pct = 0`, no `NaN`). Los duplicados se miden por
`url_producto`, que es el identificador único real, **no** por `nombre`: hay 17 nombres
repetidos que son presentaciones distintas del mismo producto (p. ej. la misma agua en
botellas de distinto tamaño y precio), y deduplicar por nombre las borraría por error.

In [2]:
print("nulos por columna (%):")
print((df_crudo.isnull().mean() * 100).round(1))
print("\nfilas totales:", len(df_crudo))
print("duplicados por url_producto:", df_crudo["url_producto"].duplicated().sum())
print("nombres repetidos (NO son duplicados, son presentaciones distintas):",
      int(df_crudo["nombre"].duplicated().sum()))

nulos por columna (%):
sku                 0.0
nombre              0.0
marca               0.0
categoria           0.0
precio_venta        0.0
precio_regular      0.0
tiene_precio_oh     0.0
descuento_pct       0.0
url_producto        0.0
url_imagen          0.0
fecha_extraccion    0.0
dtype: float64

filas totales: 1507
duplicados por url_producto: 0
nombres repetidos (NO son duplicados, son presentaciones distintas): 17


### 12.2 Defectos que el 0 % de nulos esconde

Tres defectos reales, ninguno visible como celda vacía:

1. **`sku` no es un SKU.** Contiene el *slug* de la URL (`galletas-soda-field-bolsa-6un`).
   Documentarlo como "SKU" en el diccionario de datos sería un error factual; se renombra a
   `id_producto`.
2. **El 15.5 % de `url_imagen` no es la foto del producto** sino el octógono de advertencia
   nutricional (`Alto_sodio.png`, `Alto_azucar.png`, `azucar_saturadas.png`…). El extractor
   toma el primer `<img>` de la tarjeta y en productos con sellos nutricionales ese es el
   octógono. Esas imágenes se sirven desde el host `vivanda.vteximg`, mientras que las fotos
   reales vienen de `plazavea.vteximg`: eso permite detectarlas sin ambigüedad.
3. **La categoría `limpieza` trae solo 15 productos** frente a ~150 de las demás. La
   extracción falló ahí (ese slug cae en el vertical de *marketplace*, con clases CSS
   distintas). Incluirla sesgaría cualquier comparación entre categorías.

In [3]:
print("productos por categoría:")
print(df_crudo["categoria"].value_counts())

es_octogono = ~df_crudo["url_imagen"].str.contains("plazavea.vteximg", na=False)
print(f"\nurl_imagen que apunta al octógono (no al producto): "
      f"{es_octogono.sum()} filas = {es_octogono.mean()*100:.1f} %")

productos por categoría:
categoria
abarrotes           150
bebidas             150
lacteos             150
cuidado-personal    150
frutas-verduras     150
carnes-pescados     150
desayunos           150
quesos-fiambres     149
panaderia           147
congelados          146
limpieza             15
Name: count, dtype: int64

url_imagen que apunta al octógono (no al producto): 233 filas = 15.5 %


## 13. Pipeline de limpieza

Ocho pasos. Cada uno registra en la **bitácora** (sección 14) qué se decidió, cuántas filas
había antes y cuántas quedan después. Se trabaja sobre una copia (`df`) para no alterar el
crudo, que se conserva como evidencia.

In [4]:
df = df_crudo.copy()

# --- Bitácora: la evidencia de control de calidad ---
bitacora = []
# Se llama DESPUÉS de aplicar cada cambio; toma el conteo de filas de `df`
# en ese momento como 'filas_despues' y lo compara con el paso anterior.
def log(paso, decision):
    antes = log.filas_previas
    despues = len(df)
    bitacora.append({
        "paso": paso,
        "decision": decision,
        "filas_antes": antes,
        "filas_despues": despues,
        "delta": despues - antes,
    })
    log.filas_previas = despues
    print(f"[{paso}] {decision}  |  filas: {antes} -> {despues} ({despues-antes:+d})")
log.filas_previas = len(df)

print("estado inicial:", len(df), "filas,", df.shape[1], "columnas")

estado inicial: 1507 filas, 11 columnas


### Paso 1 — Normalización de texto

Espacios colapsados en `nombre` y `marca`, marca a MAYÚSCULAS (unifica `Field`/`FIELD`), y
`categoria` de *slug* a formato legible mediante un mapa explícito. El mapa se declara a mano
—en vez de un `title()` automático— para poner tildes correctas (`lacteos` → `Lácteos`) y
conjunciones en minúscula (`frutas-verduras` → `Frutas y Verduras`), tal como debe aparecer
en el diccionario de datos.

In [5]:
def colapsar_espacios(s):
    return re.sub(r"\s+", " ", str(s)).strip()

df["nombre"] = df["nombre"].map(colapsar_espacios)
df["marca"] = df["marca"].map(colapsar_espacios).str.upper()

MAPA_CATEGORIAS = {
    "abarrotes": "Abarrotes",
    "bebidas": "Bebidas",
    "lacteos": "Lácteos",
    "cuidado-personal": "Cuidado Personal",
    "frutas-verduras": "Frutas y Verduras",
    "carnes-pescados": "Carnes y Pescados",
    "desayunos": "Desayunos",
    "quesos-fiambres": "Quesos y Fiambres",
    "panaderia": "Panadería",
    "congelados": "Congelados",
    "limpieza": "Limpieza",
}
df["categoria"] = df["categoria"].map(MAPA_CATEGORIAS).fillna(df["categoria"])

log("1. Normalización de texto",
    "Espacios colapsados; marca a mayúsculas; categoría de slug a formato legible")

[1. Normalización de texto] Espacios colapsados; marca a mayúsculas; categoría de slug a formato legible  |  filas: 1507 -> 1507 (+0)


### Paso 2 — Renombrar `sku` → `id_producto`

La columna llamada `sku` no contiene un SKU: es el *slug* de la URL. Mantener el nombre
`sku` en el entregable induciría un error factual en el diccionario de datos. Se renombra a
`id_producto`, que describe lo que realmente es: un identificador legible derivado de la URL.

In [6]:
df = df.rename(columns={"sku": "id_producto"})
log("2. Renombrar sku -> id_producto",
    "La columna no es un SKU sino el slug de la URL; el nombre correcto es id_producto")

[2. Renombrar sku -> id_producto] La columna no es un SKU sino el slug de la URL; el nombre correcto es id_producto  |  filas: 1507 -> 1507 (+0)


### Paso 3 — Deduplicar por `url_producto`

Se elimina cualquier fila con `url_producto` repetida (el identificador único real). **No** se
deduplica por `nombre`: las presentaciones distintas del mismo producto comparten nombre pero
son filas legítimas. En este crudo no hay URLs repetidas (0 eliminadas), pero el paso queda
documentado como control.

In [7]:
df = df.drop_duplicates(subset="url_producto", keep="first").reset_index(drop=True)
log("3. Deduplicar por url_producto",
    "Duplicados por identificador único real; nunca por nombre (son presentaciones)")

[3. Deduplicar por url_producto] Duplicados por identificador único real; nunca por nombre (son presentaciones)  |  filas: 1507 -> 1507 (+0)


### Paso 4 — Eliminar la columna `url_imagen`

El 15.5 % de esta columna apunta al octógono nutricional, no a la foto del producto: 233
valores sistemáticamente incorrectos. Se elimina la columna entera.

**Alternativa rechazada:** recodificar esos 233 valores a `NaN`. Dejaría la columna con 15.5 %
de nulos y **violaría el límite del 10 %** de la rúbrica. Una columna con un sexto de sus
valores mal es menos útil que su ausencia, sobre todo cuando ninguna parte del análisis
(precios, descuentos, categorías) depende de la imagen.

In [8]:
n_octogono = int((~df["url_imagen"].str.contains("plazavea.vteximg", na=False)).sum())
df = df.drop(columns=["url_imagen"])
log("4. Eliminar columna url_imagen",
    f"{n_octogono} valores (15.5%) apuntaban al octógono nutricional, no al producto")

[4. Eliminar columna url_imagen] 233 valores (15.5%) apuntaban al octógono nutricional, no al producto  |  filas: 1507 -> 1507 (+0)


### Paso 5 — Tipado de precios

`precio_venta` y `precio_regular` a numérico. Un precio ≤ 0 no tiene sentido de negocio: se
convierte a `NaN` (dato faltante honesto) en vez de conservar un cero engañoso. En este crudo
no hay precios ≤ 0, así que no se genera ningún nulo, pero el control queda registrado.

In [9]:
for col in ["precio_venta", "precio_regular"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df.loc[df[col] <= 0, col] = pd.NA
    df[col] = pd.to_numeric(df[col], errors="coerce")

log("5. Tipado de precios",
    "precio_venta y precio_regular a numérico; valores <= 0 pasan a nulo")

[5. Tipado de precios] precio_venta y precio_regular a numérico; valores <= 0 pasan a nulo  |  filas: 1507 -> 1507 (+0)


### Paso 6 — Coherencia de precios

Por definición el precio regular no puede ser menor que el precio de venta (con oferta, el de
venta baja; sin oferta, son iguales). Si aparece `precio_regular < precio_venta` se iguala al
de venta. En este crudo hay 0 casos, pero el chequeo protege contra un error de extracción.

In [10]:
incoherentes = (df["precio_regular"] < df["precio_venta"]).sum()
df.loc[df["precio_regular"] < df["precio_venta"], "precio_regular"] = df["precio_venta"]
log("6. Coherencia de precios",
    f"precio_regular < precio_venta igualado al de venta ({int(incoherentes)} casos)")

[6. Coherencia de precios] precio_regular < precio_venta igualado al de venta (0 casos)  |  filas: 1507 -> 1507 (+0)


### Paso 7 — Eliminar categorías con menos de 50 productos

La categoría `Limpieza` tiene 15 productos por una extracción incompleta, frente a ~150 de
las demás. Mantenerla sesgaría cualquier comparación entre categorías (un promedio de precio
sobre 15 productos no es comparable con uno sobre 150). Se descartan las categorías por debajo
de un umbral de 50 productos.

In [11]:
UMBRAL_CATEGORIA = 50
conteo = df["categoria"].value_counts()
categorias_chicas = conteo[conteo < UMBRAL_CATEGORIA].index.tolist()
df = df[~df["categoria"].isin(categorias_chicas)].reset_index(drop=True)
log("7. Eliminar categorías pequeñas",
    f"Descartadas por extracción incompleta (<{UMBRAL_CATEGORIA} productos): {categorias_chicas}")

[7. Eliminar categorías pequeñas] Descartadas por extracción incompleta (<50 productos): ['Limpieza']  |  filas: 1507 -> 1492 (-15)


### Paso 8 — Variables derivadas

Se recalcula `descuento_pct` desde los precios ya validados (no se confía en el valor que
traía el crudo) y se agregan variables de análisis:

- **`descuento_pct`** = `(1 − precio_venta / precio_regular) × 100`, 0 si no hay oferta.
- **`tiene_descuento`** (bool) = hay oferta activa.
- **`ahorro_soles`** = `precio_regular − precio_venta`.
- **`fecha_extraccion`** convertida a `datetime`.

In [12]:
hay_oferta = df["precio_regular"] > df["precio_venta"]
df["descuento_pct"] = 0.0
df.loc[hay_oferta, "descuento_pct"] = (
    (1 - df["precio_venta"] / df["precio_regular"]) * 100
).round(2)
df["tiene_descuento"] = hay_oferta
df["ahorro_soles"] = (df["precio_regular"] - df["precio_venta"]).round(2)
df["fecha_extraccion"] = pd.to_datetime(df["fecha_extraccion"], errors="coerce")

log("8. Variables derivadas",
    "descuento_pct recalculado; + tiene_descuento, ahorro_soles; fecha a datetime")

[8. Variables derivadas] descuento_pct recalculado; + tiene_descuento, ahorro_soles; fecha a datetime  |  filas: 1492 -> 1492 (+0)


## 14. Bitácora de limpieza

Registro completo de las decisiones. Es la evidencia que defiende el criterio *Calidad de los
datos* ante el docente: cada transformación con su justificación y su impacto en el número de
filas.

In [13]:
df_bitacora = pd.DataFrame(bitacora)
df_bitacora

,paso,decision,filas_antes,filas_despues,delta
0,1. Normalización de texto,Espacios colapsados; marca a mayúsculas; categ...,1507,1507,0
1,2. Renombrar sku -> id_producto,La columna no es un SKU sino el slug de la URL...,1507,1507,0
2,3. Deduplicar por url_producto,Duplicados por identificador único real; nunca...,1507,1507,0
3,4. Eliminar columna url_imagen,233 valores (15.5%) apuntaban al octógono nutr...,1507,1507,0
4,5. Tipado de precios,precio_venta y precio_regular a numérico; valo...,1507,1507,0
5,6. Coherencia de precios,precio_regular < precio_venta igualado al de v...,1507,1507,0
6,7. Eliminar categorías pequeñas,Descartadas por extracción incompleta (<50 pro...,1507,1492,-15
7,8. Variables derivadas,"descuento_pct recalculado; + tiene_descuento, ...",1492,1492,0


## 15. Verificación de la rúbrica

Tabla de nulos antes/después y comprobación explícita de los cuatro requisitos duros,
impresa como booleanos.

In [14]:
comp_nulos = pd.DataFrame({
    "nulos_%_crudo": (df_crudo.isnull().mean() * 100).round(2),
}).join(
    pd.DataFrame({"nulos_%_limpio": (df.isnull().mean() * 100).round(2)}),
    how="outer",
)
print("Nulos por columna, antes vs después (columnas eliminadas quedan como NaN):")
comp_nulos

Nulos por columna, antes vs después (columnas eliminadas quedan como NaN):


,nulos_%_crudo,nulos_%_limpio
ahorro_soles,NaN,0.0
categoria,0.0,0.0
descuento_pct,0.0,0.0
fecha_extraccion,0.0,0.0
id_producto,NaN,0.0
marca,0.0,0.0
nombre,0.0,0.0
precio_regular,0.0,0.0
precio_venta,0.0,0.0
sku,0.0,NaN


In [15]:
max_nulos = (df.isnull().mean() * 100).max()
dups = int(df["url_producto"].duplicated().sum())

checks = {
    "> 1000 registros":            len(df) > 1000,
    ">= 6 atributos":              df.shape[1] >= 6,
    "<= 10% nulos por columna":    max_nulos <= 10,
    "sin duplicados (url_producto)": dups == 0,
}

print(f"registros           : {len(df)}")
print(f"atributos           : {df.shape[1]}")
print(f"máx. % nulos columna: {max_nulos:.2f}")
print(f"duplicados url      : {dups}")
print("-" * 40)
for req, ok in checks.items():
    print(f"{'OK ' if ok else 'FALLA'}  {req}: {ok}")
print("-" * 40)
print("TODOS LOS REQUISITOS:", all(checks.values()))

registros           : 1492
atributos           : 12
máx. % nulos columna: 0.00
duplicados url      : 0
----------------------------------------
OK   > 1000 registros: True
OK   >= 6 atributos: True
OK   <= 10% nulos por columna: True
OK   sin duplicados (url_producto): True
----------------------------------------
TODOS LOS REQUISITOS: True


## 16. Perfil del dataset limpio

Retrato del resultado: distribución por categoría, diversidad de marcas, rango de precios y
comportamiento de los descuentos.

In [16]:
print("productos por categoría:")
print(df["categoria"].value_counts(), "\n")
print("marcas distintas :", df["marca"].nunique())
print("rango de precios : S/ {:.2f} - S/ {:.2f}".format(
    df["precio_venta"].min(), df["precio_venta"].max()))
print("productos con descuento: {} ({:.1f} %)".format(
    int(df["tiene_descuento"].sum()), df["tiene_descuento"].mean() * 100))

productos por categoría:
categoria
Abarrotes            150
Bebidas              150
Lácteos              150
Cuidado Personal     150
Frutas y Verduras    150
Carnes y Pescados    150
Desayunos            150
Quesos y Fiambres    149
Panadería            147
Congelados           146
Name: count, dtype: int64 

marcas distintas : 272
rango de precios : S/ 0.90 - S/ 999.00
productos con descuento: 715 (47.9 %)


In [17]:
print("descuento promedio y ahorro promedio por categoría (solo con oferta):")
con_oferta = df[df["tiene_descuento"]]
resumen = con_oferta.groupby("categoria").agg(
    productos_con_oferta=("descuento_pct", "size"),
    descuento_pct_prom=("descuento_pct", "mean"),
    ahorro_soles_prom=("ahorro_soles", "mean"),
).round(2).sort_values("descuento_pct_prom", ascending=False)
resumen

descuento promedio y ahorro promedio por categoría (solo con oferta):


,productos_con_oferta,descuento_pct_prom,ahorro_soles_prom
categoria,,,
Cuidado Personal,111,29.97,83.89
Congelados,53,17.47,3.31
Frutas y Verduras,30,16.44,0.95
Quesos y Fiambres,84,16.12,2.27
Desayunos,94,14.71,2.76
Abarrotes,84,14.57,1.63
Panadería,61,13.59,1.17
Carnes y Pescados,37,12.12,2.96
Bebidas,75,9.95,0.84


## 17. Limitaciones del dataset

Estas limitaciones nacen de cómo se extrajeron los datos (secciones 1–10) y acotan qué
análisis son válidos. Declararlas es parte del control de calidad y del *Alcance* del informe.

- **Muestreo por tope.** La extracción usó `max_productos = 150` por categoría, así que el
  dataset no es el catálogo completo sino los primeros ~150 productos que el sitio muestra en
  cada categoría, en el orden por defecto. **No es una muestra aleatoria**, por lo que no debe
  extrapolarse al catálogo entero de plazaVea.
- **Foto de un momento.** Los precios son del **16/09/2026**. plazaVea cambia promociones con
  frecuencia; el dataset es un corte transversal y **no permite análisis temporal**.
- **Dependencia de la zona de entrega.** El catálogo y los precios de plazaVea varían según la
  ubicación configurada en el sitio. Los datos corresponden a la **zona por defecto, sin sesión
  iniciada**.
- **Categoría `Limpieza` excluida** por extracción incompleta (15 de ~150 productos). Las
  conclusiones no cubren esa categoría.
- **Sin columna de imagen.** Se eliminó `url_imagen` por no ser confiable (15.5 % apuntaba al
  octógono nutricional). El dataset no soporta análisis que dependa de la foto del producto.

## 18. Exportación y columnas finales

Se exporta a `data/processed/plazavea_limpio.csv` en UTF-8. **Las 12 columnas de abajo quedan
congeladas**: el diccionario de datos parte de esta lista.

In [18]:
OUT.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT, index=False, encoding="utf-8")

print("exportado a:", OUT)
print("forma final:", df.shape)
print("\ncolumnas finales congeladas para el diccionario de datos:")
for i, (col, tipo) in enumerate(df.dtypes.items(), 1):
    print(f"  {i:2d}. {col:<18} {tipo}")

exportado a: ..\data\processed\plazavea_limpio.csv
forma final: (1492, 12)

columnas finales congeladas para el diccionario de datos:
   1. id_producto        str
   2. nombre             str
   3. marca              str
   4. categoria          str
   5. precio_venta       float64
   6. precio_regular     float64
   7. tiene_precio_oh    bool
   8. descuento_pct      float64
   9. url_producto       str
  10. fecha_extraccion   datetime64[us]
  11. tiene_descuento    bool
  12. ahorro_soles       float64
